Study the objectives for different attack objectives, L2 and cos distance.

In [3]:
import sys
import os

# Manually specify the path to your project folder
project_path = os.path.abspath('..')  # Or provide the full path if needed
sys.path.append(project_path)

# Now you can import from the sibling folder
from utilities import *
from helper import *

import torch.nn as nn
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.transforms as T

import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from models import *
import random
from collections import Counter
from collections import OrderedDict
import seaborn as sns
import copy

cos = nn.CosineSimilarity(dim=0, eps=1e-9) 
device = "cuda"

# study 1 model's training with the new loss function with dist limits

# adjustable parameters
# alpha_d = 100 # IID
iters = 25
# n_clients = 100 # dataset size for one client

global attack 
attack = "untargeted" #"backdoor", "tlp", "ut"
model_name = "ConvNet" # "resnet8", "ConvNet"
num_classes = 10
dataset ="fmnist" # "cifar10", "fmnist"

In [4]:
# Define transformation (convert images to tensors and normalize)
transform_img = T.Compose([
    T.ToTensor(),  # Convert image to tensor
    T.Normalize((0.5,), (0.5,))  # Normalize the image with mean and std
])

if dataset == "mnist":
    train_data = datasets.MNIST(root=f'{project_path}/data', train=True, download=True, transform=transform_img)
    test_data = datasets.MNIST(root=f'{project_path}/data', train=False, download=True, transform=transform_img)
elif dataset == "fmnist":
    train_data = datasets.FashionMNIST(root=f'{project_path}/data', train=True, download=True, transform=transform_img)
    test_data = datasets.FashionMNIST(root=f'{project_path}/data', train=False, download=True, transform=transform_img)
elif dataset == "cifar10":
    train_data = datasets.CIFAR10(root=f'{project_path}/data', train=True, download=True, transform=transform_img)
    test_data = datasets.CIFAR10(root=f'{project_path}/data', train=False, download=True, transform=transform_img)

In [9]:
# Create DataLoader for batch processing
fl_data_dict, labels_per_client_dict, label_count_per_client_dict =\
    create_data_fl_clients(client_num=100,
                           noniid_type="dirichlet",
                           label_skew_alpha =None,
                           p_degree=None,
                           dirichlet_alpha=100,
                           fl_train=train_data,
                           fl_class_num=10)
client_loader = DataLoader(fl_data_dict[0], batch_size=32, shuffle=True, num_workers=0)

In [6]:
label_count_per_client_dict[0]

{0: 52, 1: 65, 2: 55, 3: 59, 4: 67, 5: 61, 6: 62, 7: 65, 8: 71, 9: 54}

In [10]:
model_class = get_model(model_name)    
model_fn = partial(model_class, num_classes=num_classes, dataset=dataset)
# model0
model0 = model_fn().to(device)
model0_sd = {k: v.clone().detach() for k, v in model0.state_dict().items()}
optimizer0 = optim.SGD(model0.parameters(), lr=0.001)
c_criterion0 = nn.CrossEntropyLoss()

# model1 
model1 = model_fn().to(device)
model1.load_state_dict(model0_sd)
optimizer1 = optim.SGD(model1.parameters(), lr=0.001)
c_criterion1 = nn.CrossEntropyLoss()

In [11]:
# normal training to converage
for iter in range(iters):
    train_c_model(client_loader, model0, c_criterion0, optimizer0, attack_method='none', client_id=0,
                    malicious_client_ids=[], optimizer_poison=None, dba_poisoning_per_batch=0.2, device=device)
    
    test_loss, accuracy, per_class_accuracy, attack_success_rate = fl_validation(model0, c_criterion0, 
                                test_data, server_batch_size=32, class_num=10, attack_method='none',
                                mali_labels=[0,7], device=device)
    print(iter, accuracy)

0 0.3952
1 0.5962
2 0.6093
3 0.6577
4 0.6707
5 0.6767
6 0.6978
7 0.7119
8 0.7135
9 0.7162
10 0.7051
11 0.7178
12 0.727
13 0.7239
14 0.7315
15 0.7299
16 0.7252
17 0.7328
18 0.73
19 0.7357
20 0.7368
21 0.7383
22 0.7403
23 0.7395
24 0.7449
